# SS3 Part B - Applying the Final SS2 Method to Client Data

**Client:** STADIOEquities 
**Task:** Fraud detection & AML Alert triage
**Method (from SS2):** Random Forest classifier with SMOTE on the trianing set only, evaluated with imbalance-aware metrics

---
## Component 1: Dataset Loading 


In [3]:
import pandas as pd
import numpy as np

# Load the core labelled table - this table contains the target (investigation_disposition)
alerts = pd.read_csv("../data/raw_ss3/table1_compliance_alerts.csv")

# The target has three classes; check the distribution
print("Total alerts loaded:", len(alerts))
print("\nTarget (investigation_disposition) distribution:")
print(alerts["investigation_disposition"].value_counts())
print("\nProportions:")
print(alerts["investigation_disposition"].value_counts(normalize=True).round(4))

Total alerts loaded: 2864

Target (investigation_disposition) distribution:
investigation_disposition
False positive    2236
True positive      351
Unresolved         277
Name: count, dtype: int64

Proportions:
investigation_disposition
False positive    0.7807
True positive     0.1226
Unresolved        0.0967
Name: proportion, dtype: float64


### Dataset loading — explanation

**Source and format.** The data is loaded from `table1_compliance_alerts.csv`, the
core labelled table from the client extract. It is supplied as CSV, one row per
compliance alert, which is the natural format for this tabular, row-per-record
problem and matches how the client's monitoring system exports alerts. The other
six tables in the extract provide supporting context and are joined in during
preprocessing; this table is loaded first because it is the only one containing
the target label.

**Number of instances.** There are **2,864 alerts** in total.

**Classes.** The target, `investigation_disposition`, has three values:

| Class | Count | Proportion |
|---|---|---|
| False positive | 2,236 | 78.1% |
| True positive | 351 | 12.3% |
| Unresolved | 277 | 9.7% |

**Imbalance.** The problem is imbalanced. Once the 277 **Unresolved** alerts are
removed (they carry no known outcome and cannot be used as training labels), the
task becomes a **binary classification** of False positive versus True positive,
with roughly **86% False positive and 14% True positive**. This imbalance is
central to the project: the dominant class is false positives, which is precisely
the burden the triage model is intended to reduce. It also means accuracy is an
unsuitable headline metric, and precision, recall, F1 and PR-AUC are used instead,
consistent with the SS2 method.